<a href="https://colab.research.google.com/github/KangKyungHwa/DSMH_2/blob/main/__10_22/%EB%B9%85%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B6%84%EC%84%9D%EC%97%B0%EC%8A%B5(10%EC%9B%9422%EC%9D%BC).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.빅데이터 분석 연습

##문제) 아파트 가격을 결정하는 10개의 독립변수와 1개의 종속변수 데이터 이용
###(1)각 독립변수의 이상치를 사분위수를 기준으로 측정하고 이상치가 가장 많은 독립변수의 이상치 개수 구하기
####(2)이 때 결측치 먼저 제거한 후 이상치를 구한다.
####(3)단, 이상치는  1Q-1.5IQR 보다 작거나  3Q+1.5IQR 보다 큰 값이며 IQR은 Q3에서 Q1을 뺀 값이다



##2. 데이터 수집 및 전처리

In [ ]:
!git clone 'https://'github.com/KangKyungHwa/DSMH_2/''

# 1. 나눔고딕 폰트 설치
!sudo apt-get update -qq
!sudo apt-get install -y fonts-nanum-extra -qq

# 2. 시스템 폰트 캐시 업데이트
!sudo fc-cache -fv

# 3. Matplotlib 폰트 캐시 삭제
!rm -rf ~/.cache/matplotlib

from matplotlib import pyplot as plt

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['font.size'] = 11

Cloning into 'DSMH_2'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 192 (delta 6), reused 2 (delta 2), pack-reused 179 (from 1)
Receiving objects: 100% (192/192), 1.71 MiB | 21.36 MiB/s, done.
Resolving deltas: 100% (69/69), done.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously

(1) 데이터 불러오기

In [ ]:
from os import setns
import pandas as pd # 데이터 로드 및 전처리에 사용
import numpy as np
import statsmodels.formula.api as smf # 통계적 회귀 모델 생성 및 분석에 사용
import seaborn as sns
import matplotlib.pyplot as plt


apt_df = pd.read_csv("DSMH_2/__10_22/apartment_prices_dataset.csv")

print(apt_df.head(), '\n\n')
print(apt_df.tail())

         Size  Bedrooms  Bathrooms  Floor  Age  DistanceToCenter  \
0  132.322026         2          2     12   23          5.916263   
1  157.278405         2          2     16    1          4.765030   
2  131.732477         1          1     18   18         16.529690   
3  113.548220         3          1     13   43         15.602367   
4  146.884117         2          1     13    7          9.115029   

   NearbySchools  NearbyParks  PublicTransport  SafetyIndex        Price  
0            7.0          1.0              5.0           54  204546.3152  
1            9.0          4.0              5.0           44  224020.0640  
2            1.0          2.0              3.0           94  110609.5388  
3            3.0          1.0              4.0           75  143313.9356  
4            6.0          3.0              1.0           89  162672.0736   


           Size  Bedrooms  Bathrooms  Floor  Age  DistanceToCenter  \
730  126.029444         2          1     19    4         17.934586  

(2) 데이터 정보확인하기

In [ ]:
print (apt_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 735 entries, 0 to 734
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Size              735 non-null    float64
 1   Bedrooms          735 non-null    int64  
 2   Bathrooms         735 non-null    int64  
 3   Floor             735 non-null    int64  
 4   Age               735 non-null    int64  
 5   DistanceToCenter  735 non-null    float64
 6   NearbySchools     735 non-null    float64
 7   NearbyParks       735 non-null    float64
 8   PublicTransport   735 non-null    float64
 9   SafetyIndex       735 non-null    int64  
 10  Price             735 non-null    float64
dtypes: float64(6), int64(5)
memory usage: 63.3 KB
None


(3) 기술통계 확인하기 :데이터의 중위수가(50%) 평균보다 지나치게 작은경우 왼쪽으로 치우치며 오른쪽으로 꼬리가 긴 데이타 분포를 보인다. 분석데이타의 Size 컬럼의 평균은 124.58 이며 중위수는(50%) 121.90으로 많은 차이는 보이지 않음

In [ ]:
print (apt_df.describe())

             Size    Bedrooms   Bathrooms       Floor         Age  \
count  735.000000  735.000000  735.000000  735.000000  735.000000   
mean   124.582571    2.571429    1.489796    9.840816   26.025850   
std     47.019547    1.228632    0.529348    5.998226   17.277685   
min     21.205359    0.000000    0.000000    0.000000    0.000000   
25%     87.131283    2.000000    1.000000    5.000000   12.000000   
50%    121.907682    3.000000    1.000000   10.000000   25.000000   
75%    160.456029    4.000000    2.000000   15.000000   38.000000   
max    563.615954   11.000000    5.000000   54.000000  130.000000   

       DistanceToCenter  NearbySchools  NearbyParks  PublicTransport  \
count        735.000000     735.000000   735.000000       735.000000   
mean          10.016021       4.448667     2.049735         5.195673   
std            5.481185       3.176804     1.487053         3.001572   
min            0.313919       0.000000     0.000000         0.332843   
25%            5.3

(4) 결측치 제거하기  

In [ ]:
apt_df1=apt_df.dropna()  #데이타프레임을 다시 만들어 사용
print(apt_df1.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 735 entries, 0 to 734
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Size              735 non-null    float64
 1   Bedrooms          735 non-null    int64  
 2   Bathrooms         735 non-null    int64  
 3   Floor             735 non-null    int64  
 4   Age               735 non-null    int64  
 5   DistanceToCenter  735 non-null    float64
 6   NearbySchools     735 non-null    float64
 7   NearbyParks       735 non-null    float64
 8   PublicTransport   735 non-null    float64
 9   SafetyIndex       735 non-null    int64  
 10  Price             735 non-null    float64
dtypes: float64(6), int64(5)
memory usage: 63.3 KB
None


(5) 모든 변수를 대상으로 이상치 구하기 (for문 활용)

In [ ]:
#이상치 구하기

#마지막 prices는 종속변수이므로 처음부터 맨 마지막 직전(-1) 요소까지 잘라낸다,
cols = apt_df1.columns[:-1]
o_cnt_per_var ={}

for i in cols:
  #하위 25% 값(Q1)과 하위 75% 값(Q3)을 구해서 각각 Q1과 Q3라는 변수에 저장
  Q1, Q3 = apt_df1[i].quantile([0.25, 0.75])
  IQR = Q3 - Q1
  o_cnt = ((apt_df1[i] < (Q1 - 1.5 * IQR)) | (apt_df1[i] > (Q3 + 1.5 * IQR))).sum()
  o_cnt_per_var[i] = o_cnt

print(o_cnt_per_var)  #각각 컬럼의 이상치 갯수 출력


{'Size': np.int64(1), 'Bedrooms': np.int64(3), 'Bathrooms': np.int64(2), 'Floor': np.int64(4), 'Age': np.int64(15), 'DistanceToCenter': np.int64(1), 'NearbySchools': np.int64(5), 'NearbyParks': np.int64(4), 'PublicTransport': np.int64(5), 'SafetyIndex': np.int64(3)}


(6) 이상치가 가장많은 컬럼 출력

In [ ]:
#key : 각 컬럼 이름이 아닌 해당하는 '이상치 개수(Value)'를 가져와서 크기를 비교

result= max(o_cnt_per_var, key=o_cnt_per_var.get)

print(result)

Age
